In [0]:
catalog = "cinedata_medallion"
land_schema_name = "landing"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

land_schema = f"{catalog}.{land_schema_name}"
bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path =  f"/Volumes/{catalog}/{land_schema_name}/inputs"

##1. Cotação dólar

###1.1 Reading bronze

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze_cotacao = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

df_diario = (
    df_bronze_cotacao
    .withColumn("data", F.to_date("dataHoraCotacao"))
    .withColumn("cotacao_compra", F.col("cotacaoCompra").cast("decimal(10,4)"))
    .filter(F.col("data").isNotNull() & F.col("cotacao_compra").isNotNull())
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("data").orderBy(F.col("dataHoraCotacao").desc())
    ))
    .filter(F.col("rn") == 1)
    .select("data", "cotacao_compra")
)

display(df_diario.orderBy("data"))

###1.2 Creating calendar

In [0]:
min_max = df_diario.agg(
    F.min("data").alias("min_data"),
    F.max("data").alias("max_data")
).collect()[0]

min_data = min_max["min_data"]
max_data = min_max["max_data"]

print(f"Time period: {min_data} to {max_data}")

calendar_df = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(F.lit(min_data), F.lit(max_data), F.expr("interval 1 day"))
        ).alias("data")
    )
)

print(f"Days in calendar: {calendar_df.count()}")

###1.3 Foward filling

In [0]:
df_silver_cotacao = (
    calendar_df
    .join(df_diario, on="data", how="left")
    .withColumn(
        "cotacao_compra_brl",
        F.last("cotacao_compra", ignorenulls=True).over(
            Window.orderBy("data").rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )
    .withColumn("processed_timestamp", F.current_timestamp())
    .select("data", "cotacao_compra_brl", "processed_timestamp")
)

display(df_silver_cotacao.orderBy("data"))

###1.4 Recording in silver

In [0]:
df_silver_cotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

print(f"{silver_schema}.tb_cotacao_dolar recorded with success.")
print(f"Line amount: {df_silver_cotacao.count()}")

###1.5 Validation

In [0]:

nulls = df_silver_cotacao.filter(F.col("cotacao_compra_brl").isNull()).count()
print(f"Lines with null price: {nulls}") 

display(
    df_silver_cotacao
    .withColumn("dia_semana", F.date_format("data", "EEEE"))
    .orderBy("data")
    .limit(20)
)

##2. Avaliações usuário

In [0]:
from pyspark.sql import functions as F

df_bronze_reviews = spark.table(f"{bronze_schema}.tb_movies_reviews")

###2.1 Apply bussiness rules

In [0]:
df_silver_reviews = (
    df_bronze_reviews
    #Deduplication
    .dropDuplicates(["id", "nome", "nota", "comentario"])

    #Validate movie review
    .withColumn("nota", F.col("nota").cast("double"))
    .withColumn(
        "nota_usuario",
        F.when(
            (F.col("nota") >= 0) & (F.col("nota") <= 10),
            F.col("nota")
        ).otherwise(F.lit(None).cast("double"))
    )

    #Apply comment business rule
    .withColumn(
        "comentario_usuario",
        F.when(
            F.col("comentario").isNull() | (F.trim(F.col("comentario")) == ""),
            F.lit("Sem comentário")
        ).otherwise(F.col("comentario"))
    )

    #Rename columns
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")

    .withColumn("processed_timestamp", F.current_timestamp())

    #Select columns
    .select("id_filme", "nome_usuario", "nota_usuario", "comentario_usuario", "processed_timestamp")
)

display(df_silver_reviews.limit(10))

###2.2 Record to silver

In [0]:
df_silver_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

print(f"{silver_schema}.tb_avaliacoes_usuarios saved with success.")

###2.3 Validation

In [0]:
duplicates = (
    df_silver_reviews
    .groupBy("id_filme", "nome_usuario", "nota_usuario", "comentario_usuario")
    .count()
    .filter("count > 1")
    .count()
)
print(f"Duplicates left: {duplicates}") 

invalid_reviews = (
    df_silver_reviews
    .filter(
        F.col("nota_usuario").isNotNull() &
        ((F.col("nota_usuario") < 0) | (F.col("nota_usuario") > 10))
    )
    .count()
)
print(f"Reviews out of bounds: {invalid_reviews}") 

null_comments = (
    df_silver_reviews
    .filter(
        F.col("comentario_usuario").isNull() |
        (F.trim(F.col("comentario_usuario")) == "")
    )
    .count()
)
print(f"Null comments: {null_comments}") 

print(f"\nTotal lines: {df_silver_reviews.count()}")
print(f"Null reviews: {df_silver_reviews.filter(F.col('nota_usuario').isNull()).count()}")

##3. Financeiro filmes

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze_financials = spark.table(f"{bronze_schema}.tb_movies_financials")

###3.1 Currency cleaning function

In [0]:
def clean_currency(col):
    cleaned = F.regexp_replace(col, r"[\$\s]|USD", "")
    
    has_m = cleaned.endswith("M")
    num = F.when(
        has_m,
        F.regexp_replace(cleaned, "M$", "").cast("double") * 1_000_000
    ).otherwise(
        cleaned.try_cast("double")
    )
    
    return F.when(num > 0, num).otherwise(F.lit(None).cast("double"))

###3.2 Get exchange rate

In [0]:
#I have decided to choose the most recent dollar value.
#It simplifies the logic and does not affect the analysis.

df_cotacao = spark.table(f"{silver_schema}.tb_cotacao_dolar")
taxa_row = df_cotacao.orderBy(F.col("data").desc()).limit(1).collect()

if taxa_row:
    taxa_atual = float(taxa_row[0]["cotacao_compra_brl"])
    print(f"Tax used: R$ {taxa_atual:.4f} (date: {taxa_row[0]['data']})")
else:
    taxa_atual = None
    print("[WARNING] tb_cotacao_dolar empty, BRL values will be NULL.")

###3.3 Full data processing

In [0]:
df_silver_financials = (
    df_bronze_financials
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("id").orderBy(F.col("ingestion_timestamp").desc())
        #The ID deduplication was my idea, I think by doing this I can avoid future problems in next layers.
        #It also makes the table more organized and easier to read.
    ))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .filter(F.col("id").isNotNull())
    
    .withColumn("orcamento_usd", clean_currency(F.col("budget")))
    .withColumn("receita_usd", clean_currency(F.col("revenue")))
)

# BRL conversion
if taxa_atual is not None:
    df_silver_financials = (
        df_silver_financials
        .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.lit(taxa_atual)).cast("decimal(18,2)"))
        .withColumn("receita_brl", (F.col("receita_usd") * F.lit(taxa_atual)).cast("decimal(18,2)"))
    )
else:
    df_silver_financials = (
        df_silver_financials
        .withColumn("orcamento_brl", F.lit(None).cast("decimal(18,2)"))
        .withColumn("receita_brl", F.lit(None).cast("decimal(18,2)"))
    )

# Decimal + derivadas cast
df_silver_financials = (
    df_silver_financials
    .withColumn("orcamento_usd", F.col("orcamento_usd").cast("decimal(18,2)"))
    .withColumn("receita_usd", F.col("receita_usd").cast("decimal(18,2)"))
    
    # Profit
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)"))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)"))
    
    # Margin
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            (F.col("receita_usd").isNotNull()) & (F.col("receita_usd") > 0),
            F.round((F.col("lucro_usd") / F.col("receita_usd")) * 100, 2)
        ).otherwise(F.lit(None).cast("double"))
    )
    
    # Renaming and timestamp addition
    .withColumnRenamed("id", "id_filme")
    .withColumn("processed_timestamp", F.current_timestamp())
    
    # Columns selection
    .select(
        "id_filme",
        "orcamento_usd", "receita_usd",
        "orcamento_brl", "receita_brl",
        "lucro_usd", "lucro_brl", "margem_lucro_percentual",
        "processed_timestamp"
    )
)

print(f"Total lines after processing: {df_silver_financials.count()}")
display(df_silver_financials.limit(10))

###3.4 Record in silver

In [0]:
df_silver_financials.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

print(f"{silver_schema}.tb_financeiro_filmes saved with success.")

###3.5 Validation

In [0]:
dupes = df_silver_financials.groupBy("id_filme").count().filter("count > 1").count()
print(f"id_filme dupes: {dupes}")

zero_neg = df_silver_financials.filter(
    (F.col("orcamento_usd") <= 0) | (F.col("receita_usd") <= 0)
).count()
print(f"zero/negative values: {zero_neg}") 

display(df_silver_financials.select(
    F.count("*").alias("total"),
    F.count("orcamento_usd").alias("com_orcamento"),
    F.count("receita_usd").alias("com_receita"),
    F.count("lucro_usd").alias("com_lucro"),
    F.count("margem_lucro_percentual").alias("com_margem"),
    F.round(F.avg("margem_lucro_percentual"), 2).alias("margem_media_pct")
))

##4. Info filmes

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze_movies_info = spark.table(f"{bronze_schema}.tb_movies_info")
print(f"Total: {df_bronze_movies_info.count()}")

###4.1 Normalization & status translation function

In [0]:
def normalize_and_translate_status(col):
    normalized = F.lower(F.trim(col))
    normalized = F.regexp_replace(normalized, "-", " ")
    normalized = F.regexp_replace(normalized, r"\s+", " ")
    
    return (
        F.when(normalized == "released", "Lançado")
        .when(normalized == "post production", "Pós-Produção")
        .when(normalized == "in production", "Em Produção")
        .when(normalized == "planned", "Planejado")
        .when(normalized == "rumored", "Rumores")
        .when(normalized == "canceled", "Cancelado")
        .otherwise("Não Informado")
    )

###4.2 Date conversion function

In [0]:
#Will try ISO first, after that, it is going to be really hard to determine what date is brazilian or american.
# In this case I defined american as the priority. This is usually the expected course of action to take in this case. 

def parse_multi_date(col):
    return F.coalesce(
        F.try_to_date(col, "yyyy-MM-dd"),   # ISO 
        F.try_to_date(col, "MM-dd-yyyy"),   # american
        F.try_to_date(col, "dd/MM/yyyy")    # brazilian
    )

###4.3 full table processing

In [0]:
df_silver_movies_info = (
    df_bronze_movies_info

    #Deduplication
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("id").orderBy(F.col("ingestion_timestamp").desc())
    ))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .filter(F.col("id").isNotNull())
    
    #Date conversion
    .withColumn("data_lancamento", parse_multi_date(F.col("release_date")))
    
    #ano_lancamento = data_lancamento year
    .withColumn("ano_lancamento", F.year("data_lancamento"))
    
    #if runtime == 0, transform to NULL
    #I decided to do that because it does not make sense for movie to have no runtime
    .withColumn("runtime_num", F.col("runtime").try_cast("int"))   
    .withColumn("duracao_minutos",
        F.when(F.col("runtime_num") > 0, F.col("runtime_num"))      
        .otherwise(F.lit(None).cast("int"))
    )
    .drop("runtime_num")                                            
    
    #Normalize and translate
    .withColumn("status_filme", normalize_and_translate_status(F.col("status")))

    # Synopsis cleaning
    .withColumn("sinopse_limpa",
        F.trim(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(F.col("overview"), r'\\', ''),        
                    r'["\']{2,}', '"'                                       
                ),
                r'["\'][^"\']*$', ''                                        
            )
        )
    )
    
    #Rename columns
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("overview", "sinopse")
    
    # Overwrite sinopse with the cleaned version
    .withColumn("sinopse", F.col("sinopse_limpa"))
    .drop("sinopse_limpa")
    
    .withColumn("processed_timestamp", F.current_timestamp())
    
    # Selected columns
    .select(
        "id_filme", "titulo", "titulo_original",
        "data_lancamento", "ano_lancamento", "duracao_minutos",
        "idioma_original", "status_filme", "sinopse",
        "processed_timestamp"
    )
)

print(f"Lines after processing: {df_silver_movies_info.count()}")
display(df_silver_movies_info.limit(10))

###4.4 Silver recording

In [0]:
df_silver_movies_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

print(f"{silver_schema}.tb_info_filmes recorded with success.")

###4.5 Validation

In [0]:
#Check for dupes
dupes = df_silver_movies_info.groupBy("id_filme").count().filter("count > 1").count()
print(f"id_filme dupes: {dupes}")

#Check for valid statuses
valid_statuses = ["Lançado", "Pós-Produção", "Em Produção",
                   "Planejado", "Rumores", "Cancelado", "Não Informado"]
fora_dominio = df_silver_movies_info.filter(~F.col("status_filme").isin(valid_statuses)).count()
print(f"Statuses out of domain: {fora_dominio}")

#Check for null dates
null_dates = df_silver_movies_info.filter(F.col("data_lancamento").isNull()).count()
print(f"Null dates: {null_dates}")

#Check for null runtimes
null_runtimes = df_silver_movies_info.filter(F.col("duracao_minutos").isNull()).count()
print(f"Null runtimes: {null_runtimes}")

# Status distribution
display(df_silver_movies_info.groupBy("status_filme").count().orderBy(F.desc("count")))

##5. Generos

In [0]:
from pyspark.sql import functions as F

df_bronze_credits_and_tags  = spark.table(f"{bronze_schema}.tb_credits_and_tags")
print(f"Total: {df_bronze_credits_and_tags.count()}")
display(df_bronze_credits_and_tags.select("id", "genres").limit(10))

###5.1 Full table processing

In [0]:
#I chose to keep the movie ID as a column in this table.
#I think it eases the process of joining this table with the other tables later on.

TMDB_GENRES = [
    "Action", "Adventure", "Animation", "Comedy", "Crime",
    "Documentary", "Drama", "Family", "Fantasy", "History",
    "Horror", "Music", "Mystery", "Romance", "Science Fiction",
    "TV Movie", "Thriller", "War", "Western"
]

df_silver_generos = (
    df_bronze_credits_and_tags
    # ID deduplication
    .select("id", "genres")
    .dropDuplicates(["id"])
    .filter(F.col("id").isNotNull())
    .filter(F.col("genres").isNotNull())
    
    # Split + explode, covers ",;|"
    .withColumn("nome_genero", F.explode(F.split(F.col("genres"), r"[,;|]")))
    
    # Special characters removal
    .withColumn("nome_genero", F.trim(F.col("nome_genero")))
    .withColumn("nome_genero", F.regexp_replace(F.col("nome_genero"), r'["\']', ""))
    .withColumn("nome_genero", F.trim(F.col("nome_genero")))
    
    # removal filters
    .filter(F.col("nome_genero") != "")
    .filter(F.col("nome_genero").rlike("[A-Za-z]"))     
    .filter(~F.col("nome_genero").rlike(r"\.\s"))      
    .filter(F.length(F.col("nome_genero")) < 20) 
    .filter(F.col("nome_genero").isin(TMDB_GENRES)) 
    
    # Rename colunms + processed timestamp
    .withColumnRenamed("id", "id_filme")
    .withColumn("processed_timestamp", F.current_timestamp())
    .select("id_filme", "nome_genero", "processed_timestamp")
    .distinct()
)

print(f"Pair count (filme, gênero): {df_silver_generos.count()}")
print(f"Distinct genre count: {df_silver_generos.select('nome_genero').distinct().count()}")
display(df_silver_generos.limit(10))

###5.2 Silver recording

In [0]:
df_silver_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_generos")

print(f"{silver_schema}.tb_generos recorded with success.")

###5.3 Validation

In [0]:
movies_with_genres = df_silver_generos.select("id_filme").distinct().count()
print(f"Movies with genres: {movies_with_genres}")

# Most frequent genres
display(
    df_silver_generos
    .groupBy("nome_genero")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
)

suspects = df_silver_generos.filter(F.col("nome_genero").contains(",")).count()
print(f"Genres with comma: {suspects}") 

null_genres = df_silver_generos.filter(F.col("nome_genero") == "").count()
print(f"Null genres: {null_genres}")

average = df_silver_generos.groupBy("id_filme").count().agg(F.avg("count")).collect()[0][0]
print(f"Genre per movie average: {average:.2f}")

##6. Pessoas empresas

In [0]:
from pyspark.sql import functions as F

df_bronze_credits_and_tags  = spark.table(f"{bronze_schema}.tb_credits_and_tags")


###6.1 Extract entities function

In [0]:
LANG_COUNTRY_BLACKLIST = {
    "english", "japanese", "spanish", "french", "german", "italian",
    "portuguese", "russian", "chinese", "korean", "hindi", "arabic",
    "dutch", "swedish", "polish", "turkish", "thai", "vietnamese",
    "indonesian", "greek", "hebrew", "finnish", "danish", "norwegian",
    "czech", "hungarian", "romanian", "ukrainian", "catalan", "xhosa",
    "latin", "icelandic", "serbian", "croatian", "bulgarian", "slovak",
    "tagalog", "malay", "persian", "urdu", "bengali", "tamil", "telugu",
    "united states of america", "united states", "usa", "united kingdom",
    "france", "japan", "spain", "germany", "italy", "brazil", "canada",
    "australia", "china", "india", "russia", "mexico", "south korea",
    "argentina", "new zealand", "ireland", "belgium", "netherlands",
    "sweden", "norway", "denmark", "finland", "poland", "portugal",
    "american", "british", "french", "japanese", "german", "italian",
    "spanish", "portuguese", "russian", "chinese", "korean", "indian",
    "australian", "canadian", "mexican", "brazilian",
}

def extract_entities(df, column_name, tipo):
    return (
        df
        .select("id", F.col(column_name).alias("entidades_raw"))
        .dropDuplicates(["id"])
        .filter(F.col("id").isNotNull())
        .filter(F.col("entidades_raw").isNotNull())
        
        # Split + explode
        .withColumn("nome_entidade", F.explode(F.split(F.col("entidades_raw"), r"[,;|]")))
        
        # Special characters filters
        .withColumn("nome_entidade", F.trim(F.col("nome_entidade")))
        .withColumn("nome_entidade", F.regexp_replace(F.col("nome_entidade"), r'["\']', ""))
        .withColumn("nome_entidade", F.trim(F.col("nome_entidade")))
        
        # Captalization
        .withColumn("nome_entidade", F.initcap(F.col("nome_entidade")))
        
        # Removal filters
        .filter(F.col("nome_entidade") != "")
        .filter(F.col("nome_entidade").rlike("[A-Za-z]"))
        .filter(~F.lower(F.col("nome_entidade")).isin(
            "n/a", "na", "nenhum", "none", "unknown", "sem informação"
        ))
        .filter(~F.lower(F.col("nome_entidade")).isin(*LANG_COUNTRY_BLACKLIST))
        .filter(~F.col("nome_entidade").rlike(r"\.\s"))  
        .filter(~F.col("nome_entidade").rlike(r"\.(jpg|jpeg|png|gif|webp|svg)$"))  
        .filter(~F.col("nome_entidade").rlike(r"^/"))                              
        .filter(~F.col("nome_entidade").rlike(r"^[#&@%*]"))                        
        .filter(~F.col("nome_entidade").rlike(r"^\([^)]*$"))                        
        .filter(F.length(F.col("nome_entidade")) < 60)          
        
        .withColumn("tipo_entidade", F.lit(tipo))
        .select("id", "nome_entidade", "tipo_entidade")
    )

###6.2 Extract the 4 entities and unify 

In [0]:
df_atores      = extract_entities(df_bronze_credits_and_tags, "cast", "Ator")
df_diretores   = extract_entities(df_bronze_credits_and_tags, "directors", "Diretor")
df_roteiristas = extract_entities(df_bronze_credits_and_tags, "writers", "Roteirista")
df_produtoras  = extract_entities(df_bronze_credits_and_tags, "production_companies", "Produtora")

# Unify all dataframes into one
df_silver_pessoas = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
    
    .distinct()
    
    .withColumnRenamed("id", "id_filme")
    .withColumn("processed_timestamp", F.current_timestamp())
    .select("id_filme", "tipo_entidade", "nome_entidade", "processed_timestamp")
)

print(f"Pair count (filme, entidade): {df_silver_pessoas.count()}")
display(df_silver_pessoas.groupBy("tipo_entidade").count().orderBy(F.desc("count")))

###6.3 Record in silver

In [0]:
df_silver_pessoas.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

print(f"{silver_schema}.tb_pessoas_empresas recorded with success.")

###6.4 Validation

In [0]:
# Distribution by type
display(df_silver_pessoas.groupBy("tipo_entidade").count().orderBy(F.desc("count")))

#Top 20 entities per number of movies
display(
    df_silver_pessoas
    .groupBy("tipo_entidade", "nome_entidade")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
)

#Quality check
for tipo in ["Ator", "Diretor", "Roteirista", "Produtora"]:
    n = df_silver_pessoas.filter(F.col("tipo_entidade") == tipo).count()
    n_distintas = df_silver_pessoas.filter(F.col("tipo_entidade") == tipo).select("nome_entidade").distinct().count()
    print(f"{tipo}: {n} pairs | {n_distintas} distinct entities")

#Suspect finder
com_aspas = df_silver_pessoas.filter(F.col("nome_entidade").rlike(r'["\']')).count()
print(f"\nEntities with quotation: {com_aspas}")  

#Separator finder
for sep in [",", ";", "|"]:
    n = df_silver_pessoas.filter(F.col("nome_entidade").contains(sep)).count()
    print(f"Entities with '{sep}': {n}")  

##7. Métricas engajamento

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")
print(f"Total: {df_bronze_metrics.count()}")
df_bronze_metrics.printSchema()
display(df_bronze_metrics.limit(10))

###7.1 Cleaning functions

In [0]:
def clean_decimal(col):
    cleaned = F.regexp_replace(F.trim(col), ",", ".")
    return cleaned.try_cast("double")

def clean_nota(col):
    num = clean_decimal(col)
    return F.when((num >= 0) & (num <= 10), num).otherwise(F.lit(None).cast("double"))

def clean_contagem(col):
    num = clean_decimal(col)
    return F.when(num >= 0, num.cast("long")).otherwise(F.lit(None).cast("long"))

def clean_popularidade(col):
    num = clean_decimal(col)
    return F.when(
        (num >= 0) & (num <= 1000),
        num
    ).otherwise(F.lit(None).cast("double"))

###7.2 Silver processing

In [0]:
df_silver_metrics = (
    df_bronze_metrics
    # ID deduplication
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("id").orderBy(F.col("ingestion_timestamp").desc())
    ))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .filter(F.col("id").isNotNull())
    
    # Cleaning and validation
    .withColumn("popularidade", clean_popularidade(F.col("popularity")))
    .withColumn("nota_media_tmdb", clean_nota(F.col("vote_average")))
    .withColumn("qtd_votos_tmdb", clean_contagem(F.col("vote_count")))
    .withColumn("nota_media_imbd", clean_nota(F.col("averageRating")))
    .withColumn("qtd_votos_imbd", clean_contagem(F.col("numVotes")))
    
    # Rename columns and add timestamp
    .withColumnRenamed("id", "id_filme")
    .withColumn("processed_timestamp", F.current_timestamp())
    
    #Select columns
    .select(
        "id_filme",
        "popularidade",
        "nota_media_tmdb", "qtd_votos_tmdb",
        "nota_media_imbd", "qtd_votos_imbd",
        "processed_timestamp"
    )
)

print(f"Lines amount after processing: {df_silver_metrics.count()}")
display(df_silver_metrics.limit(10))

###7.3 Record in silver

In [0]:
df_silver_metrics.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

print(f"{silver_schema}.tb_metricas_engajamento recorded with success.")

###7.4 Validation

In [0]:
# Dupes check
dupes = df_silver_metrics.groupBy("id_filme").count().filter("count > 1").count()
print(f"id_filme dupes: {dupes}")

# Reviews out of range
notas_invalidas = df_silver_metrics.filter(
    (F.col("nota_media_tmdb") < 0) | (F.col("nota_media_tmdb") > 10) |
    (F.col("nota_media_imbd") < 0) | (F.col("nota_media_imbd") > 10)
).count()
print(f"Reviews out of range: {notas_invalidas}")

#Negative data
contagens_neg = df_silver_metrics.filter(
    (F.col("qtd_votos_tmdb") < 0) | (F.col("qtd_votos_imbd") < 0) |
    (F.col("popularidade") < 0)
).count()
print(f"Negative data: {contagens_neg}")

#Statistics
display(df_silver_metrics.select(
    F.count("*").alias("total"),
    F.count("popularidade").alias("com_popularidade"),
    F.count("nota_media_tmdb").alias("com_nota_tmdb"),
    F.count("qtd_votos_tmdb").alias("com_votos_tmdb"),
    F.count("nota_media_imbd").alias("com_nota_imbd"),
    F.count("qtd_votos_imbd").alias("com_votos_imbd"),
))